In [0]:
%run ./ingestion_utils

In [0]:
# BLS source configuration
BLS_BASE_URL = "https://download.bls.gov/pub/time.series/pr/"

# Unity Catalog Volume
VOLUME_PATH = "/Volumes/workspace/rearc/rearc_landing"

# Raw landing location
BLS_RAW_PATH = f"{VOLUME_PATH}/raw/bls"

# Manifest tracks file versions and checksums
BLS_MANIFEST_PATH = f"{BLS_RAW_PATH}/manifest.json"

# Source-specific configuration
REQUEST_TIMEOUT_SECONDS = 60
MAX_RETRIES = 3

print("BLS ingestion configuration loaded")
print("Source:", BLS_BASE_URL)
print("Landing path:", BLS_RAW_PATH)
print("Manifest:", BLS_MANIFEST_PATH)

In [0]:
from html.parser import HTMLParser
from urllib.parse import urljoin, urlparse
import os

class BLSDirectoryParser(HTMLParser):
    """Extract file links from the BLS directory listing."""

    def __init__(self):
        super().__init__()
        self.links = []

    def handle_starttag(self, tag, attrs):
        if tag.lower() == "a":
            href = dict(attrs).get("href")
            if href:
                self.links.append(href)


def discover_bls_files(base_url, http_session):
    response = http_session.get(
        base_url,
        timeout=REQUEST_TIMEOUT_SECONDS
    )
    response.raise_for_status()

    parser = BLSDirectoryParser()
    parser.feed(response.text)

    discovered = set()

    for href in parser.links:
        absolute_url = urljoin(base_url, href)
        parsed = urlparse(absolute_url)

        # Accept only files from the expected BLS directory.
        if parsed.netloc != urlparse(base_url).netloc:
            continue

        if parsed.path.rsplit("/", 1)[0] != urlparse(base_url).path.rstrip("/"):
            continue

        filename = os.path.basename(parsed.path)

        if filename.startswith("pr.") and filename:
            discovered.add(filename)

    return sorted(discovered)


# Create an HTTP session using our shared utility
bls_session = create_http_session()

# Discover files
bls_files = discover_bls_files(BLS_BASE_URL, bls_session)

print("Discovered BLS files:", len(bls_files))

for filename in bls_files:
    print(filename)

assert bls_files, "No BLS files discovered. Check source connectivity."

In [0]:
import inspect

print("download_file:", inspect.signature(download_file))
print("calculate_sha256:", inspect.signature(calculate_sha256))
print("load_manifest:", inspect.signature(load_manifest))
print("save_manifest:", inspect.signature(save_manifest))

In [0]:

import os
from pathlib import Path
from datetime import datetime, timezone

# --------------------------------------------------
# 1. Validate source discovery
# --------------------------------------------------

current_files = set(bls_files)

if not current_files:
    raise RuntimeError(
        "No BLS files discovered. Refusing to update the manifest."
    )

# --------------------------------------------------
# 2. Load previous ingestion state
# --------------------------------------------------

manifest = load_manifest(BLS_MANIFEST_PATH)
manifest.setdefault("files", {})
manifest.setdefault("removed_files", [])

previous_files = manifest["files"]

downloaded = []
unchanged = []
failed = []

# --------------------------------------------------
# 3. Download and process each discovered file
# --------------------------------------------------

for filename in sorted(current_files):

    source_url = f"{BLS_BASE_URL}{filename}"
    destination = f"{BLS_RAW_PATH}/{filename}"
    temp_destination = f"{destination}.tmp"

    Path(destination).parent.mkdir(
        parents=True,
        exist_ok=True
    )

    try:
        # Download to temporary location.
        # Existing landing file remains untouched on failure.
        result = download_file(
            session,
            source_url,
            temp_destination
        )

        new_hash = result["sha256"]
        new_size = result["file_size"]

        old_entry = previous_files.get(filename, {})
        old_hash = old_entry.get("sha256")

        destination_exists = os.path.exists(destination)

        # Verify the existing landing file itself.
        existing_hash = (
            calculate_sha256(destination)
            if destination_exists
            else None
        )

        # --------------------------------------------------
        # 4. Unchanged file
        # --------------------------------------------------

        if (
            old_hash == new_hash
            and existing_hash == new_hash
        ):
            os.remove(temp_destination)

            # Keep manifest metadata current on every run.
            old_entry.update({
                "sha256": new_hash,
                "size_bytes": new_size,
                "source_url": source_url,
                "status": "active",
                "last_seen_at": datetime.now(
                    timezone.utc
                ).isoformat()
            })

            previous_files[filename] = old_entry

            unchanged.append(filename)
            print(f"UNCHANGED: {filename}")

        # --------------------------------------------------
        # 5. New or changed file
        # --------------------------------------------------

        else:
            # Atomic replacement after successful download.
            os.replace(temp_destination, destination)

            previous_files[filename] = {
                "sha256": new_hash,
                "size_bytes": new_size,
                "source_url": source_url,
                "status": "active",
                "last_updated": datetime.now(
                    timezone.utc
                ).isoformat(),
                "last_seen_at": datetime.now(
                    timezone.utc
                ).isoformat()
            }

            downloaded.append(filename)
            print(f"DOWNLOADED/UPDATED: {filename}")

    except Exception as exc:

        # Remove incomplete temporary download.
        if os.path.exists(temp_destination):
            os.remove(temp_destination)

        failed.append(filename)

        print(
            f"FAILED: {filename} | "
            f"Error: {str(exc)}"
        )

# --------------------------------------------------
# 6. Detect files removed from the source
# --------------------------------------------------

removed_files = {
    filename
    for filename, entry in previous_files.items()
    if (
        filename not in current_files
        and entry.get("status") != "removed_from_source"
    )
}

for filename in sorted(removed_files):

    previous_files[filename]["status"] = (
        "removed_from_source"
    )

    print(f"REMOVED FROM SOURCE: {filename}")

# Preserve removal history without duplicates.
historical_removed = set(manifest["removed_files"])

manifest["removed_files"] = sorted(
    historical_removed.union(removed_files)
)

# --------------------------------------------------
# 7. Save manifest
# --------------------------------------------------

manifest["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

save_manifest(
    manifest,
    BLS_MANIFEST_PATH
)

# --------------------------------------------------
# 8. Ingestion summary
# --------------------------------------------------

print("\n========== BLS INGESTION SUMMARY ==========")

print("Discovered:", len(current_files))
print("Downloaded/Updated:", len(downloaded))
print("Unchanged:", len(unchanged))
print("Failed:", len(failed))
print("Removed from source:", len(removed_files))

print("Manifest:", BLS_MANIFEST_PATH)

# Fail the notebook/job if any download failed.
if failed:
    raise RuntimeError(
        f"BLS ingestion failed for {len(failed)} file(s): "
        f"{failed}"
    )

print("\nBLS INGESTION COMPLETED SUCCESSFULLY")


In [0]:
import os

manifest = load_manifest(BLS_MANIFEST_PATH)
manifest_files = manifest["files"]

validation_results = []

for filename in sorted(bls_files):
    file_path = f"{BLS_RAW_PATH}/{filename}"

    exists = os.path.exists(file_path)
    entry = manifest_files.get(filename, {})

    actual_size = os.path.getsize(file_path) if exists else 0
    expected_size = entry.get("size_bytes")

    valid = (
        exists
        and entry.get("status") == "active"
        and actual_size == expected_size
        and bool(entry.get("sha256"))
    )

    validation_results.append(valid)

    print(
        filename,
        "| exists:", exists,
        "| size:", actual_size,
        "| manifest size:", expected_size,
        "| valid:", valid
    )

assert all(validation_results), "BLS landing validation failed!"

print("\nBLS LANDING VALIDATION PASSED")
print("Validated files:", len(validation_results))

In [0]:
manifest = load_manifest(BLS_MANIFEST_PATH)

print("Manifest keys:", manifest.keys())
print("\nFirst file entry:")
print(manifest["files"]["pr.class"])

In [0]:
manifest = load_manifest(BLS_MANIFEST_PATH)

for filename in bls_files:
    if filename in manifest["files"]:
        manifest["files"][filename]["status"] = "active"

manifest["updated_at"] = datetime.now(timezone.utc).isoformat()

save_manifest(manifest, BLS_MANIFEST_PATH)

print("Manifest status updated successfully")
print("Files marked active:", len(bls_files))